# JalVaani v4.2: honest prediction intervals + Pune fix (post-processing only), with simple results

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

**Upload in Step 1** (where each file is on your laptop):

| file | what it is |
|---|---|
| `tft_model_v3.py` | `03_models/tft/`: the TFT network (unchanged) |
| `tft_v4.py` | `03_models/tft/`: the pipeline, **pipeline 4.6 or newer** (= v4.2) |
| `master_dataset_v4target.csv` | `data/master/`: the dataset (25 districts, 2023-01-01 → 2026-06-04) |
| `RESULTS_GUIDE.md` | `docs/`: the guide; it is copied into the results zip |
| `tft_metrics_per_district.csv` | *optional*: the OLD v1 file from `04_results/tft_prod_v1/`, for a side-by-side table |

**What v4.2 changes.** Only what happens *after* the models have predicted. The TFT networks and LightGBM are the v4.1 ones, so the forecast medians stay the same apart from small GPU run-to-run noise.
1. **Interval calibration.** v4.1's adaptive conformal step (ACI) made the TFT's 80% intervals *worse*: coverage went from 77–80% to 66–74% on the locked test. Two causes were found in the code:
   - every TFT cell fed back its own outcome, about 10 correlated outcomes per issue date;
   - the α limits (0.01–0.30 around 0.10) left more room to narrow than to widen.

   v4.2 tries fixes for both, and then chooses, **per model and horizon**, whether to report the calibrated or the network's own interval.
2. **Pune guard.** Pune's Ensemble leaned on LightGBM, which is poor there. Where the stack is worse than both the seasonal forecast and the TFT on a district's own past errors, that district now uses the better of the two.

**What it does (about 45 min):**
1. Trains dev folds 1–4 **once**, with the v4.1 settings and 10 seeds. Every later dev variant reuses these networks, at no training cost.
2. Applies three rules, **fixed before the run**, on dev folds 1–4 only: **G** (Pune guard), **C** (conformal variant), **D** (calibrated or raw, per model × horizon). Folds 5–6 are not even computed until step 3.
3. **Final run**, which trains folds 5–6. Reports all 6 folds and the locked folds 5–6. Also reports the **v4.1 post-processing on the same networks**, so the v4.1 → v4.2 comparison is like-for-like.
4. **Simple results** (v1 style) for all 24 districts, for the Ensemble and the TFT alone.
5. Downloads `tft_v4_2_outputs.zip`. Read `RESULTS_GUIDE.md` inside it.

In [ ]:
# ── STEP 1: upload the files listed above (from your laptop) ─────────────────
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
!pip install -q lightgbm

In [ ]:
# ── STEP 2: setup ────────────────────────────────────────────────────────────
import sys, time, shutil, zipfile, os, json
from pathlib import Path
import numpy as np, pandas as pd, torch
from IPython.display import Image, display, Markdown
sys.path.insert(0, '/content')
import tft_v4 as v4

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again'
assert tuple(map(int, v4.PIPELINE_VERSION.split('.'))) >= (4, 6), \
    f'old tft_v4.py ({v4.PIPELINE_VERSION}) - upload the v4.2 one (pipeline 4.6)'
print('GPU:', torch.cuda.get_device_name(0), '| pipeline', v4.PIPELINE_VERSION)

DATA = '/content/master_dataset_v4target.csv'
OUT  = Path('/content/outputs/v4_2')
OUT.mkdir(parents=True, exist_ok=True)
assert Path(DATA).exists(), 'master_dataset_v4target.csv was not uploaded'
T0 = time.time()
def elapsed(): return (time.time() - T0) / 60
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

In [ ]:
# ── STEP 3: settings ─────────────────────────────────────────────────────────
SEEDS = [42, 1337, 2026, 7, 99, 123, 555, 777, 2024, 31337]          # the v4.1 final seeds

# The v4.1 final set-up: s1_tab2, stack_prior_n 500, aci_gamma 0.01. Networks and LightGBM are unchanged in v4.2.
# refit is off: v4.1 trained a refit pass but did not use it (use_refit=False), and pass 1 does not depend on it,
# so leaving it out only saves time.
REF_KW = dict(target_mode='snaive', calib_mode='v4', combine='stack',
              train_stride=1, epoch_frac=1/7, use_refit=False,
              tab_features='v2', tab_stride=1, tab_fresh=True, gbm_n_estimators=400, gbm_lr=0.03, gbm_min_child=100,
              stack_prior_n=500, aci_gamma=0.01)

# Rule C candidates (V0 = v4.1). Listed in simplicity order.
VARIANTS = {
    'V0_v4.1':           dict(aci_feedback='cell',   aci_gamma=0.01, aci_alpha_min=0.01, aci_alpha_max=0.30),
    'V1_anchor':         dict(aci_feedback='anchor', aci_gamma=0.01, aci_alpha_min=0.01, aci_alpha_max=0.30),
    'V2_anchor_sym':     dict(aci_feedback='anchor', aci_gamma=0.01, aci_alpha_min=0.00, aci_alpha_max=0.20),
    'V3_anchor_sym_g03': dict(aci_feedback='anchor', aci_gamma=0.03, aci_alpha_min=0.00, aci_alpha_max=0.20),
    'V4_static':         dict(aci_feedback='cell',   aci_gamma=0.00, aci_alpha_min=0.01, aci_alpha_max=0.30),
}

def make(tag, mode='dev', **kw):
    cfg = v4.Config(tag=tag, mode=mode, data_path=DATA, out_root=str(OUT), seeds=list(SEEDS))
    for k, v in {**REF_KW, **kw}.items():
        setattr(cfg, k, v)
    return cfg

def run_dev(tag, **kw):
    # dev folds 1-4 only; after the first call the networks and LightGBM come from the cache
    return v4.run(make(tag, mode='dev', **kw), verbose=False)

def brief(tag, res):
    S = v4.report(res, verbose=False, save_predictions=False)
    rows = []
    for name, sm in S.items():
        h, c = sm['head'], sm['cov']
        rows.append({'run': tag, 'model': name,
                     **{f'vs no-change {k}': h.loc[k, 'rMAE_P'] for k in v4.HORDER},
                     **{f'vs seasonal {k}': h.loc[k, 'rMAE_SN'] for k in v4.HORDER},
                     **{f'cov {k}': c.loc[k, 'Cov (%)'] for k in v4.HORDER},
                     **{f'interval score {k}': c.loc[k, 'Interval score (m)'] for k in v4.HORDER},
                     'mean vs seasonal': h['rMAE_SN'].mean()})
    return pd.DataFrame(rows)

## Step A: train dev folds 1–4 once (v4.1 settings, 10 seeds)
This is the only network training on folds 1–4. Every variant below reuses these networks and LightGBM models from the cache, and changes only post-processing. Folds 5–6 (the locked test) are not computed until the final run.

In [ ]:
print(f'Training dev folds 1-4 with {len(SEEDS)} seeds ...', flush=True)
REF_DEV = run_dev('dev_ref')
B_REF = brief('ref = v4.1 post-processing', REF_DEV)
display(B_REF.round(3))
print('For orientation only: v4.1 dev screening (3 seeds) had Ensemble mean vs seasonal 0.939; '
      'expect a value within about 0.01 of that')
print(f'[{elapsed():.0f} min elapsed]')

## Rule G: the Pune guard (`stack_guard`)
The guard acts per district and horizon, using only that district's own pool of past out-of-sample errors (at least 50 weighted rows). If the stacked forecast was worse there than **both** seasonal-naive alone and the TFT alone, the district uses the better of the two.

**Adopt it if, on dev folds 1–4 (21 headline districts):**
- the mean Ensemble error ÷ seasonal-naive error over t+7, t+14 and t+30 does **not get worse**; **and**
- no district's Ensemble MASE rises by more than **0.02** at any horizon.

In [ ]:
GUARD_DEV = run_dev('dev_guard', stack_guard=True)
ADOPT_GUARD, m_off, m_on, GTAB = v4.guard_decision(REF_DEV, GUARD_DEV)
GTAB.to_csv(OUT / 'guard_decision.csv', index=False)
if len(GUARD_DEV['guarded']):
    GUARD_DEV['guarded'].to_csv(OUT / 'guard_cases_dev.csv', index=False)
print(f'dev mean Ensemble vs seasonal: guard off {m_off:.4f} | on {m_on:.4f} | '
      f'largest district MASE rise {GTAB["change"].max():+.3f}')
display(Markdown('**Guarded (district, horizon, fold) on the dev folds**')); display(GUARD_DEV['guarded'])
display(Markdown('**Pune, Ensemble MASE on the dev folds**')); display(GTAB[GTAB.district == 'Pune'])
print('ADOPT stack_guard:', ADOPT_GUARD)
GUARD_KW = {'stack_guard': ADOPT_GUARD}
BASE_DEV = GUARD_DEV if ADOPT_GUARD else REF_DEV

## Rule C: which conformal (calibration) variant
| variant | ACI feedback | γ | α range |
|---|---|---|---|
| V0 (= v4.1) | every cell | 0.01 | 0.01–0.30 |
| V1 | one outcome per issue date and band | 0.01 | 0.01–0.30 |
| V2 | one per issue date and band | 0.01 | 0.00–0.20 (symmetric round 0.10) |
| V3 | one per issue date and band | 0.03 | 0.00–0.20 |
| V4 | none (static conformal, γ = 0) | 0 | – |

**Score** = the mean over {TFT, Ensemble} × {t+7, t+14, t+30} of (calibrated interval score ÷ that model and horizon's un-calibrated interval score). Below 1 means calibration helps.

**Pick** the lowest score. V0 is kept unless another variant beats it by more than **0.01**.

In [ ]:
VAR_ROWS, VAR_TABLES = [], {}
for name, kw in VARIANTS.items():
    r = BASE_DEV if name == 'V0_v4.1' else run_dev(f'dev_{name}', **GUARD_KW, **kw)
    t = v4.interval_table(r)
    VAR_TABLES[name] = t
    cal = t[t.interval == 'calibrated'].set_index(['model', 'horizon'])
    VAR_ROWS.append({'variant': name, **kw, 'score (cal / raw)': v4.calibration_score(t),
                     **{f'{m} cov {h}': cal.loc[(m, h), 'Cov (%)'] for m in ('TFT', 'Ensemble') for h in v4.HORDER}})
    print(f'{name}: score {VAR_ROWS[-1]["score (cal / raw)"]:.4f}   [{elapsed():.0f} min]', flush=True)
VAR = pd.DataFrame(VAR_ROWS)
VAR.to_csv(OUT / 'conformal_variant_summary.csv', index=False)
display(VAR.round(3))
sc = VAR.set_index('variant')['score (cal / raw)']
CHOSEN_VAR = sc.idxmin() if sc.min() < sc['V0_v4.1'] - 0.01 else 'V0_v4.1'
print('Chosen conformal variant:', CHOSEN_VAR)

## Rule D: report the calibrated or the network's own interval, per model × horizon
Done on dev folds 1–4, for the chosen variant:
- if exactly one of the two has coverage within **75–85%**, take it;
- otherwise take the lower interval score;
- if they are within 1%, take the un-calibrated one (the simpler option).

In [ ]:
ITAB = VAR_TABLES[CHOSEN_VAR]
INTERVAL_CHOICE, DECIDE = v4.pick_interval_choice(ITAB)
ITAB.to_csv(OUT / 'interval_choice_table.csv', index=False)
DECIDE.to_csv(OUT / 'interval_choice_decisions.csv', index=False)
display(DECIDE.round(3))
FINAL_KW = {**GUARD_KW, **VARIANTS[CHOSEN_VAR], 'interval_choice': INTERVAL_CHOICE}
print('Final v4.2 post-processing:', FINAL_KW, f'  [{elapsed():.0f} min elapsed]')

## Final run: all 6 folds
This trains folds 5–6; folds 1–4 come from the cache. Results are reported twice:
- all 6 folds;
- folds 5–6 only, the locked test, which none of the rules above looked at.

The **v4.1 post-processing on these same networks** is computed too, at no cost. That makes the v4.1 → v4.2 comparison like-for-like.

In [ ]:
FINAL_TAG = 'FINAL_v4_2'
res = v4.run(make(FINAL_TAG, mode='all', **FINAL_KW), verbose=True)
print('\n\n================  ALL 6 FOLDS  ================')
S_all = v4.report(res, verbose=True)

In [ ]:
print('================  LOCKED TEST: folds 5-6 (Oct 2025 -> Jun 2026)  ================')
locked = v4.subset_folds(res, list(v4.LOCKED_FOLDS), FINAL_TAG + '_locked')
S_locked = v4.report(locked, verbose=True, save_predictions=False)
QUOTE = v4.results_to_quote(S_locked, S_all)
QUOTE.to_csv(OUT / 'RESULTS_TO_QUOTE.csv', index=False)

# the v4.1 post-processing on the SAME networks (no training: everything is cached)
ref = v4.run(make('REF_v4_1_postproc', mode='all'), verbose=False)
R_all = v4.report(ref, verbose=False, save_predictions=False)
R_locked = v4.report(v4.subset_folds(ref, list(v4.LOCKED_FOLDS), 'REF_v4_1_postproc_locked'),
                     verbose=False, save_predictions=False)
REFQ = v4.results_to_quote(R_locked, R_all)
REFQ.to_csv(OUT / 'RESULTS_TO_QUOTE_ref_v4_1_postproc.csv', index=False)

# published v4.1 numbers (Colab run of 2026-10-29, calibrated intervals for both models)
V41 = pd.DataFrame([
    (t, m, h, mae, p, s, c, isc) for (t, m), vals in {
        ('locked folds 5-6', 'Ensemble'): [(0.3140, .8392, .9384, 81.1, 1.676), (0.4059, .7627, .9221, 80.1, 2.057), (0.5501, .6162, .8766, 80.0, 3.062)],
        ('locked folds 5-6', 'TFT'):      [(0.3264, .8725, .9756, 74.2, 2.010), (0.4276, .8034, .9713, 70.0, 2.490), (0.6023, .6746, .9597, 65.7, 3.557)],
        ('all 6 folds', 'Ensemble'):      [(0.4291, .9031, .9666, 79.9, 2.403), (0.5953, .8301, .9488, 78.4, 3.606), (0.8605, .7118, .9092, 78.1, 5.387)],
        ('all 6 folds', 'TFT'):           [(0.4440, .9346, 1.0003, 73.0, 2.740), (0.6286, .8764, 1.0018, 68.6, 3.984), (0.9236, .7640, .9758, 64.3, 5.767)],
    }.items() for h, (mae, p, s, c, isc) in zip(v4.HORDER, vals)],
    columns=['test set', 'model', 'horizon', 'v4.1 MAE (m)', 'v4.1 vs no-change', 'v4.1 vs seasonal', 'v4.1 coverage',
             'v4.1 interval score'])
cols = {'MAE (m)': 'MAE (m)', 'rMAE vs persistence': 'vs no-change', 'rMAE vs seasonal': 'vs seasonal',
        'coverage 80% (%)': 'coverage', 'interval score (m)': 'interval score', 'interval': 'interval'}
def tagged(q, pre):
    return q[['test set', 'model', 'horizon'] + list(cols)].rename(columns={k: f'{pre} {v}' for k, v in cols.items()})
CMP = V41.merge(tagged(REFQ, 'same-net v4.1-post'), on=['test set', 'model', 'horizon']) \
         .merge(tagged(QUOTE, 'v4.2'), on=['test set', 'model', 'horizon'])
CMP['v4.2 - same-net: vs no-change'] = (CMP['v4.2 vs no-change'] - CMP['same-net v4.1-post vs no-change']).round(4)
CMP['v4.2 - same-net: interval score'] = (CMP['v4.2 interval score'] - CMP['same-net v4.1-post interval score']).round(3)
CMP.to_csv(OUT / 'v4_1_vs_v4_2.csv', index=False)
display(Markdown('### v4.1 → v4.2 (lower ratios and interval scores are better)')); display(CMP.round(3))
display(Markdown('### Results to quote (v4.2)')); display(QUOTE.round(3))

# Pune, Ensemble MASE (all folds): published v4.1 1.000 / 1.050 / 1.128
PUNE = pd.concat([R_all['Ensemble']['per_dist'].assign(run='same-net v4.1-post'),
                  S_all['Ensemble']['per_dist'].assign(run='v4.2')])
PUNE = PUNE[PUNE.district == 'Pune'][['run', 'Horizon', 'rMAE_P', 'rMAE_SN', 'MAE (m)']]
PUNE.to_csv(OUT / 'pune_v4_1_vs_v4_2.csv', index=False)
display(Markdown('### Pune, Ensemble, all folds (published v4.1 MASE: 1.000 / 1.050 / 1.128)')); display(PUNE.round(3))

## Simple results (v1 style): all 24 districts, Ensemble and TFT (reported intervals)

In [ ]:
SIMPLE = OUT / 'simple'
for model in ('Ensemble', 'TFT'):
    v4.export_simple(res, model, SIMPLE / model.lower())
v4.export_shared(res, SIMPLE)
if Path('/content/tft_metrics_per_district.csv').exists():
    v4.compare_v1('/content/tft_metrics_per_district.csv', SIMPLE)
    print('v1 comparison written (NOT like-for-like - see its caveat column)')
if Path('/content/RESULTS_GUIDE.md').exists():
    shutil.copy('/content/RESULTS_GUIDE.md', OUT / 'RESULTS_GUIDE.md')
else:
    print('RESULTS_GUIDE.md was not uploaded - the zip will not contain the guide')

lines = ['# JalVaani v4.2 run summary', '',
         f'- pipeline {v4.PIPELINE_VERSION}, data {Path(DATA).name} (md5 {v4.file_md5(DATA)[:8]})',
         f'- networks / LightGBM: the v4.1 final set-up {REF_KW}',
         f'- seeds: {SEEDS}', f'- total run time: {elapsed():.0f} min',
         '- all three rules used dev folds 1-4 only; folds 5-6 are the locked test', '',
         f'## Rule G (Pune guard): adopted = {ADOPT_GUARD}',
         f'dev mean Ensemble vs seasonal: off {m_off:.4f}, on {m_on:.4f}; largest district rise {GTAB["change"].max():+.3f}', '',
         f'## Rule C (conformal variant): {CHOSEN_VAR} = {VARIANTS[CHOSEN_VAR]}', '```',
         VAR.round(3).to_string(index=False), '```', '',
         '## Rule D (reported interval per model and horizon)', '```', DECIDE.round(3).to_string(index=False), '```', '',
         '## v4.1 -> v4.2', '```', CMP.round(3).to_string(index=False), '```', '',
         '## Pune (Ensemble, all folds)', '```', PUNE.round(3).to_string(index=False), '```']
(OUT / 'RUN_SUMMARY.md').write_text('\n'.join(lines), encoding='utf-8')
print(f'done   [{elapsed():.0f} min elapsed]')

In [ ]:
# ── the key results, shown here ──────────────────────────────────────────────
display(Markdown('### Reported interval per model and horizon')); display(DECIDE[['model', 'horizon', 'pick', 'reason']])
for model in ('ensemble', 'tft'):
    display(Markdown(f'### {model.upper()}: headline (v1-style)'))
    display(pd.read_csv(SIMPLE / model / f'{model}_metrics_headline.csv').round(3))
display(Markdown('### Ensemble: MASE per district (error ÷ no-change error; below 1 is better)'))
display(pd.read_csv(SIMPLE / 'ensemble' / 'ensemble_mase_by_district.csv').round(3))
for f in ['ensemble/ensemble_per_district_mase_t30.png', 'ensemble/ensemble_coverage_heatmap.png',
          'tft/tft_coverage_heatmap.png', 'ensemble/ensemble_forecast_grid_t30.png',
          'tft/tft_per_district_mase_t30.png', 'tft_training_curves.png', 'tft_vsn_importance.png',
          'ensemble/quantile/quantile_pune.png', 'ensemble/quantile/quantile_nagpur.png']:
    p = SIMPLE / f
    if p.exists():
        display(Markdown(f'**{f}**')); display(Image(filename=str(p), width=1100))

In [ ]:
# ── zip everything except the model caches, and download ────────────────────
# the dev / reference runs reuse the final run's networks, so only FINAL_v4_2 keeps its .pt files
zpath = '/content/tft_v4_2_outputs.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and '_cache' not in p.parts and (p.suffix != '.pt' or FINAL_TAG in p.parts):
            z.write(p, p.relative_to(OUT))
print(f'{zpath}: {os.path.getsize(zpath)/1e6:.1f} MB')
files.download(zpath)